# A2 — cluster survival at the pre-registered *k* only

**Gate:** multivariate log-rank *p* < 0.05 at frozen *k*. Failure is an honest descriptive product state.
**Sensitivity sweep** is exploratory and is never a UI significance claim.


In [ ]:
from pathlib import Path
import sys, json, warnings
warnings.filterwarnings("ignore")

cwd = Path.cwd().resolve()
for cand in [cwd, *cwd.parents]:
    if (cand / "src" / "gate.py").is_file():
        sys.path.insert(0, str(cand / "src"))
        break
    nested = cand / "v2"
    if (nested / "src" / "gate.py").is_file():
        sys.path.insert(0, str(nested / "src"))
        break

from paths import ensure_src_on_path, resolve_v2_root
from gate import gate as _gate_impl
from safety import assert_safe

V2_ROOT = resolve_v2_root()
ensure_src_on_path(V2_ROOT)
REPO_ROOT = V2_ROOT.parent
RAW = V2_ROOT / "data" / "raw"
INTERIM = V2_ROOT / "data" / "interim"
V3 = INTERIM / "v3"
REF = V2_ROOT / "data" / "reference"
ARTIFACTS = V2_ROOT / "artifacts"
FIGURES = V2_ROOT / "reports" / "figures" / "v3"
for d in (RAW, INTERIM, V3, REF, ARTIFACTS, FIGURES):
    d.mkdir(parents=True, exist_ok=True)

SMOKE_TEST = False

def cohort_ids():
    import pandas as pd
    assign = V3 / "cluster_assignments.parquet"
    if assign.is_file():
        return pd.read_parquet(assign)["patient_id"].astype(str).str[:12].unique().tolist()
    expr = INTERIM / "intrinsic_expression.parquet"
    if expr.is_file():
        return pd.read_parquet(expr).index.astype(str).str[:12].unique().tolist()
    return None

def gate(*args, **kwargs):
    kwargs.setdefault("smoke_test", SMOKE_TEST)
    if "sample_ids" not in kwargs:
        ids = cohort_ids()
        if ids is not None:
            kwargs["sample_ids"] = ids
            kwargs.setdefault("n", len(ids))
            kwargs.setdefault("cohort", True)
    return _gate_impl(*args, **kwargs)

print("V2_ROOT =", V2_ROOT, "SMOKE_TEST =", SMOKE_TEST)


In [ ]:
import numpy as np
import pandas as pd
from cluster_selection import config_id
from survival_export import curves_by_cluster, multivariate_logrank, sensitivity_logrank

preg = json.loads((REF / "preregistered_k.json").read_text())
assign_path = V3 / "cluster_assignments.parquet"
clinical_candidates = list((RAW / "tcga_brca").glob("**/data_clinical_patient.txt")) if (RAW / "tcga_brca").exists() else []

if assign_path.is_file() and clinical_candidates:
    assign = pd.read_parquet(assign_path)
    clin = pd.read_csv(clinical_candidates[0], sep="\t", comment="#")
    # best-effort OS / PFI mapping
    rename = {}
    for a, b in [("OS_MONTHS", "os_months"), ("OS_STATUS", "os_status"), ("PFI_MONTHS", "pfi_months"), ("PFI_STATUS", "pfi_status")]:
        if a in clin.columns:
            rename[a] = b
    clin = clin.rename(columns=rename)
    pid_col = "PATIENT_ID" if "PATIENT_ID" in clin.columns else clin.columns[0]
    clin["patient_id"] = clin[pid_col].astype(str).str[:12]
    k = preg.get("k")
    sub = assign[(assign["method"] == "gmm") & (assign["covariance_type"] == "full") & (assign["k"] == k)]
    merged = sub.merge(clin, on="patient_id", how="inner")
    def status_to_event(s):
        return s.astype(str).str.contains("1:DECEASED|1:Event|1:", case=False).astype(float)
    if "os_months" in merged.columns:
        times_os = pd.to_numeric(merged["os_months"], errors="coerce")
        events_os = status_to_event(merged["os_status"]) if "os_status" in merged.columns else pd.Series(1.0, index=merged.index)
        labels = merged["cluster"].to_numpy()
        used_real = True
    else:
        used_real = False
else:
    used_real = False

if not used_real:
    print("A2: clinical join failed — not synthesizing. Gate fails; framing=descriptive.")
    (V3 / "survival_stats.json").write_text(json.dumps({
        "k": preg.get("k") if "preg" in dir() else None,
        "p_os": 1.0, "p_pfi": None, "n": 0, "n_events": 0,
        "framing": "descriptive", "passed": False, "source": "unavailable",
    }, indent=2))
    p_os = 1.0
else:
    os_res = multivariate_logrank(times_os, labels, events_os)
    p_os = os_res["p_value"]
    curves = curves_by_cluster(times_os.to_numpy(), events_os.to_numpy(), labels)
    rows = [{"config_id": config_id("gmm", "full", int(k)), "endpoint": "os", "cluster": cl, "exploratory": False,
             "p_value": p_os, "curve": json.dumps(curve)} for cl, curve in curves.items()]
    pd.DataFrame(rows).to_parquet(V3 / "km_curves.parquet")
    by_k = {}
    for k_i, part in assign[assign["method"].eq("gmm") & assign["covariance_type"].eq("full")].groupby("k"):
        m = part.merge(clin, on="patient_id", how="inner")
        by_k[int(k_i)] = m["cluster"].to_numpy()
        # align times — use the preregistered merge length as a fallback
    (V3 / "survival_stats.json").write_text(json.dumps({
        "k": k, "p_os": p_os, "p_pfi": None, "n": os_res["n"], "n_events": os_res["n_events"],
        "framing": "prognostic" if p_os < 0.05 else "descriptive", "passed": p_os < 0.05,
    }, indent=2))
    pd.DataFrame(sensitivity_logrank(times_os, events_os, {int(k): labels})).to_parquet(V3 / "survival_sensitivity.parquet")

print("p_os", p_os)


In [ ]:
stats = json.loads((V3 / "survival_stats.json").read_text())
gate("NB_A2", "cluster_logrank_os", float(stats.get("p_os") or 1.0), 0.05, direction="lte",
     note=f"k={stats.get('k')} n={stats.get('n')} events={stats.get('n_events')} framing={stats.get('framing')}")
if not stats.get("passed"):
    print("A2 failed: clusters are descriptive, not prognostic. Do not retune k.")
